In [2]:
import os
import re
import requests
from openai import OpenAI
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = os.getenv("OPENAI_BASE_URL")
MODEL_ID = os.getenv("OPENAI_MODEL_ID")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

AGENT_SYSTEM_PROMPT = """
你是一个智能旅行助手。你的任务是分析用户的请求，并使用可用工具一步步地解决问题。

# 可用工具:
- `get_weather(city: str)`: 查询指定城市的实时天气
- `get_attraction(city: str, weather: str)`: 根据城市和天气搜索推荐的旅游景点。

# 输出格式要求:
你的每次回复必须严格遵循以下格式，包含一对Thought和Action：

Thought: [你的思考过程和下一步计划]
Action: [你要执行的具体行动]

Action的格式必须是以下之一：
1. 调用工具：function_name(arg_name="arg_value")
2. 结束任务：Finish[最终答案]

# 重要提示：
- 每次只输出一对Thought和Action
- Action必须在同一行，不要换行
- 当收集到足够信息可以回答用户问题时，必须使用Action: Finish[最终答案]格式结束

请开始吧！
"""

In [3]:
def get_weather(city: str) -> str:
    url = f"https://wttr.in/{city}?format=j1"

    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        current_condition = data['current_condition'][0]
        weather_desc = current_condition['weatherDesc'][0]['value']
        temp_c = current_condition['temp_C']

        return f"{city}当前天气：{weather_desc}，气温{temp_c}摄氏度"
    
    except requests.exceptions.RequestException as e:
        return f"错误：无法获取{city}的天气信息。{str(e)}"
    except (KeyError, IndexError) as e:
        return f"错误：解析{city}的天气信息时发生问题。{str(e)} " 
    
def get_attraction(city: str, weather: str) -> str:
    api_key = os.environ.get("TAVILY_API_KEY")

    if not api_key:
            return "错误：Tavily API密钥未设置。"
    
    tavily = TavilyClient(api_key=api_key)

    query = f"'{city}'在'{weather}'天气下最值得去的旅游景点推荐及理由"

    try:
        response = tavily.search(query=query, search_depth="basic", include_answer=True)

        if response.get("answer"):
            return response["answer"]
        
        formatted_results = []
        for result in response.get("results", []):
            formatted_results.append(f"- {result['title']}: {result['content']}")

        if not formatted_results:
            return f"没有找到关于{city}在{weather}天气下的旅游景点推荐。"

        return "根据搜索，为您找到以下信息：\n" + "\n".join(formatted_results)

    except Exception as e:
        return f"错误：在使用Tavily搜索时发生问题。{str(e)}"   
    
available_tools = {
   "get_weather": get_weather,
    "get_attraction": get_attraction
}

    

In [4]:
class OpenAICompatibleClient:
    def __init__(self, model: str, api_key: str, base_url: str):
        self.model = model
        self.client = OpenAI(api_key=api_key, base_url=base_url)

    def generate(self, prompt: str, system_prompt: str) -> str:
        print("正在调用大语言模型...")
        try:
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ]
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                stream=False
            )
            answer = response.choices[0].message.content
            print("大语言模型响应")
            return answer
        except Exception as e:
            print(f"调用大语言模型时发生错误: {str(e)}")
            return "错误：无法生成响应，请稍后再试。"
        
class TravelAssistant:
    def __init__(self):
        self.llm = OpenAICompatibleClient(
            model=MODEL_ID,
            api_key=API_KEY,
            base_url=BASE_URL
        )
        self.prompt_history = []

    def reset(self):
        self.prompt_history = []

    def add_user_message(self, message: str):
        self.prompt_history.append(f"用户请求：{message}")

    def add_assistant_message(self, message: str):
        self.prompt_history.append(message)

    def add_observation(self, observation: str):
        self.prompt_history.append(f"Observation: {observation}")

# user_input = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"
# assistant = TravelAssistant()
# assistant.add_user_message(user_input)
# full_prompt = "\n".join(assistant.prompt_history)
# llm_output = assistant.llm.generate(full_prompt, AGENT_SYSTEM_PROMPT)
# llm_output
        

In [5]:
def display_conversation(history):
    """美观地显示对话历史"""
    print("\n" + "="*60)
    print("📝 对话历史")
    print("="*60)
    
    for i, message in enumerate(history, 1):
        if message.startswith("用户请求:"):
            print(f"\n👤 用户 [{i}]: {message[5:]}")
        elif message.startswith("Thought:"):
            print(f"\n🤔 思考 [{i}]: {message[8:].strip()}")
        elif message.startswith("Action:"):
            print(f"🛠️  行动 [{i}]: {message[7:].strip()}")
        elif message.startswith("Observation:"):
            print(f"📊 观察 [{i}]: {message[12:].strip()}")
        else:
            print(f"💬 消息 [{i}]: {message}")
    
    print("="*60 + "\n")

def parse_action(action_str):
    if action_str.startswith("Finish"):
        match = re.match(r"\w+\[(.*)\]", action_str)
        if match:
            return "finish", {"answer": match.group(1)}
        return "finish", {"answer": "任务完成"}
    
    tool_name_match = re.search(r"(\w+)\(", action_str)
    if not tool_name_match:
        return None, {}
    
    tool_name = tool_name_match.group(1)
    args_match = re.search(r"\((.*)\)", action_str)
    if args_match:
        args_str = args_match.group(1)
        kwargs = dict(re.findall(r'(\w+)="([^"]*)"', args_str))
    else:
        kwargs = {}
    
    return tool_name, kwargs

In [6]:
def run_assistant(user_input, max_iterations=5, display=True):
    assistant = TravelAssistant()
    assistant.add_user_message(user_input)

    if display:
        print(f"👤 用户输入: {user_input}")
        print("="*50)

    for i in range(max_iterations):
        if display:
            print(f"\n🔄 循环 {i+1}/{max_iterations}")

        full_prompt = "\n".join(assistant.prompt_history)
        llm_output = assistant.llm.generate(full_prompt, AGENT_SYSTEM_PROMPT)

        match = re.search(r'(Thought:.*?Action:.*?)(?=\n\s*(?:Thought:|Action:|Observation:)|\Z)', llm_output, re.DOTALL)
        if match:
            truncated = match.group(1).strip()
            if truncated != llm_output.strip():
                llm_output = truncated
                print("⚠️ 已截断多余的 Thought-Action 对")

        assistant.add_assistant_message(llm_output)

        if display:
            print(f"模型输出：\n{llm_output}")

        action_match = re.search(r"Action: (.*)", llm_output, re.DOTALL)
        if not action_match:
            observation = "错误：未能解析到Action字段。请确保你的回复严格遵循 'Thought: ... Action: ...' 的格式。"
            observation_str = f"Observation: {observation}"
            print(f"{observation_str}\n" + "="*50)
            assistant.prompt_history.append(observation_str)
            continue

        action_str = action_match.group(1).strip()
        tool_name, kwargs = parse_action(action_str)

         # 处理完成行动
        if tool_name == "finish":
            final_answer = kwargs.get("answer", "任务完成")
            if display:
                print(f"🎉 任务完成!")
                print(f"📋 最终答案: {final_answer}")
            return final_answer, assistant.prompt_history
        
        # 处理工具调用
        if tool_name in available_tools:
            if display:
                print(f"🛠️  调用工具: {tool_name}({kwargs})")
            observation = available_tools[tool_name](**kwargs)
        else:
            observation = f"错误：未定义的工具 '{tool_name}'"
        
        # 记录观察结果
        if display:
            print(f"📊 观察结果: {observation}")
            print("="*50)
        
        assistant.add_observation(observation)
    
    # 如果达到最大循环次数仍未完成
    timeout_answer = "抱歉，经过多次尝试仍未完成您的请求。请尝试简化您的问题或稍后重试。"
    if display:
        print(f"⏰ 达到最大循环次数: {timeout_answer}")
    
    return timeout_answer, assistant.prompt_history



In [7]:
def test_basic_example():
    print("开始测试北京天气+景点推荐示例")
    user_input = "你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。"

    final_answer, history = run_assistant(user_input, display=True)

    print("\n" + "="*60)
    print("📊 测试完成!")
    print("="*60)
    print(f"最终答案: {final_answer}")

    # 显示完整对话历史
    display_conversation(history)
    
    return final_answer, history


final_answer, history = test_basic_example()

开始测试北京天气+景点推荐示例
👤 用户输入: 你好，请帮我查询一下今天北京的天气，然后根据天气推荐一个合适的旅游景点。

🔄 循环 1/5
正在调用大语言模型...
大语言模型响应
模型输出：
Thought: 用户想要查询北京的天气并根据天气推荐景点。第一步我需要先调用get_weather工具查询北京的实时天气。
Action: get_weather(city="北京")
🛠️  调用工具: get_weather({'city': '北京'})
📊 观察结果: 北京当前天气：Cloudy，气温16摄氏度

🔄 循环 2/5
正在调用大语言模型...
大语言模型响应
模型输出：
Thought: 已经获取到北京的天气情况为Cloudy（多云）。接下来我需要根据这个天气情况和城市，调用get_attraction工具来为用户推荐合适的旅游景点。
Action: get_attraction(city="北京", weather="Cloudy")
🛠️  调用工具: get_attraction({'city': '北京', 'weather': 'Cloudy'})
📊 观察结果: In cloudy weather, visit the Longcheng Scenic Area for a picturesque experience, explore the Lama Temple for its serene atmosphere, and enjoy the cultural exhibits at the National Museum of China.

🔄 循环 3/5
正在调用大语言模型...
大语言模型响应
模型输出：
Thought: 已经获取到北京的天气和推荐的景点信息，现在可以整理信息并回复用户，结束任务。
Action: Finish[您好！今天北京的天气是多云，气温大约16摄氏度。在这种舒适的天气下，为您推荐以下几个合适的旅游景点：1. 龙城风景区：风景如画，非常适合游览；2. 雍和宫：气氛宁静祥和，适合静心参观；3. 中国国家博物馆：可以尽情享受各种文化展览。祝您旅途愉快！]
🎉 任务完成!
📋 最终答案: 您好！今天北京的天气是多云，气温大约16摄氏度。在这种舒适的天气下，为您推荐以下几个合适的旅游景点：1. 龙城风景区：